# 05 `fmiss` Model Progression

This notebook is the counterexample chapter. It shows that the strategy that works best on `fpos` does not simply transfer to `fmiss`, and that the best `fmiss` path stayed more context-first and more conservative.

Unless otherwise noted, the benchmark tables below report **20 recording-disjoint seeds (`42` to `61`)** and the diagnostics pool the saved holdout predictions across those splits.

**Questions answered here**
- Which `fmiss` modeling path actually worked?
- Why did many `fpos`-inspired tricks fail to transfer?
- How broad were the winning `fmiss` gains across families and recordings?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("05_fmiss_model_progression")
benchmark = build_benchmark_progress_table("fmiss", scope="core")
ablation_winners = build_ablation_winner_table("fmiss")
seed_ids = ["fmiss_paired_raw", "fmiss_paired_context", "fmiss_reduced_latent"]
seed_story = build_seed_rank_bundle("fmiss", seed_ids)
wave_ids = ["fmiss_paired_raw", "fmiss_hybrid_only", "fmiss_paired_context", "fmiss_reduced_latent"]
waves = build_main_wave_summary("fmiss", wave_ids)
lofo_family = build_fmiss_lofo_main_family_preference_table()
winner_diag = build_prediction_diagnostics("fmiss_reduced_latent", "fmiss")


## 1. Benchmark ladder

The main-text `fmiss` ladder is also compressed to the core narrative set. The point is not to show every intermediate variant, but to make the central result legible: `fmiss` rewards a more conservative context-first path and does not benefit from transplanting the strongest `fpos` waveform recipe.


In [ ]:
display(benchmark[["label", "mae", "rmse", "r2", "bias", "calibration_slope", "delta_r2_vs_basic_transfer", "delta_mae_vs_basic_transfer", "notes"]])
save_table(benchmark, table_dir, "fmiss_benchmark_ladder")
fig, _ = plot_benchmark_metric(benchmark, metric="r2", title="`fmiss` benchmark ladder by R² (20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fmiss_benchmark_r2")
fig


In [ ]:
fig, _ = plot_benchmark_metric(benchmark, metric="mae", title="`fmiss` benchmark ladder by MAE (20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fmiss_benchmark_mae")
fig


In [ ]:
show_saved_figure(
    fig_dir / "fmiss_seed_stability.png",
    "The `fmiss` stability view shows why this target is the weaker transfer story: the central tendency stays much closer to break-even and the split-to-split variance is much larger.",
)
show_saved_figure(
    fig_dir / "fmiss_seed_rank_heatmap.png",
    "This rank heatmap complements the violin plot by showing which of the three main compared `fmiss` models actually wins each of the 20 recording-disjoint seeds.",
)


In [ ]:
display(seed_story["summary"])
save_table(seed_story["summary"], table_dir, "fmiss_seed_rank_summary")
save_table(seed_story["winner_rows"], table_dir, "fmiss_seed_winner_by_split")


In [ ]:
import matplotlib.pyplot as plt

rank_pivot = seed_story["rank_pivot"].copy()
fig, ax = plt.subplots(figsize=(8.6, 2.9))
im = ax.imshow(rank_pivot.to_numpy(dtype=float), cmap="YlGn_r", vmin=1, vmax=max(len(rank_pivot), 1), aspect="auto")
ax.grid(False)
ax.set_xticks(range(rank_pivot.shape[1]))
ax.set_xticklabels(rank_pivot.columns.astype(str).tolist(), rotation=0)
ax.set_yticks(range(rank_pivot.shape[0]))
ax.set_yticklabels(rank_pivot.index.tolist())
ax.set_xlabel("Recording-disjoint seed")
ax.set_title("Rank by seed across the three main compared `fmiss` models\n(20 recording-disjoint seeds, 42-61; 1 = best `R²`)", pad=10)
for row_idx, row_label in enumerate(rank_pivot.index):
    for col_idx, col_label in enumerate(rank_pivot.columns):
        value = rank_pivot.loc[row_label, col_label]
        if pd.notna(value):
            ax.text(col_idx, row_idx, int(value), ha="center", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Rank")
fig.tight_layout()
save_figure(fig, fig_dir, "fmiss_seed_rank_heatmap")
fig


In [ ]:
winner_row = benchmark.sort_values("r2", ascending=False).iloc[0]
paired_context_row = benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_context"].iloc[0]
hybrid_only_row = benchmark.loc[benchmark["recipe_id"] == "fmiss_hybrid_only"].iloc[0]
display(Markdown(
    f"""
## Main `fmiss` result

- The best `fmiss` row is **{winner_row['label']}** with **R² {winner_row['r2']:.4f}**.
- Compared with the source-only transfer baseline, the retained best row improves from **R² {hybrid_only_row['r2']:.4f}** to **R² {winner_row['r2']:.4f}**.
- Compared with the simpler paired-context model, the reduced-latent stack adds a smaller but still meaningful gain (**R² {paired_context_row['r2']:.4f}** to **{winner_row['r2']:.4f}**).
"""
))


## 2. `fmiss` ablation story

The ablations here are organized around `fmiss`-specific scientific questions: source target choice, context, transfer family, unlabeled paired use, and representation/filtering.


In [ ]:
display(ablation_winners)
save_table(ablation_winners, table_dir, "fmiss_ablation_winners")
for group_name in ["source_target_choice", "context", "transfer_family"]:
    bundle = build_ablation_bundle("fmiss", group_name, "recording_disjoint_main")
    display(Markdown(f"### {group_name.replace('_', ' ').title()}"))
    display(bundle["benchmark"][["label", "mae", "r2", "notes"]])
    save_table(bundle["benchmark"], table_dir, f"fmiss_ablation_{group_name}")


## 3. Family-held-out behavior

The stricter `fmiss` family story should be read under leave-one-family-out evaluation rather than ordinary recording-disjoint family averaging. This section keeps the comparison aligned with the retained main compared `fmiss` models: `Paired-only raw`, `Paired-only context`, and `Reduced-latent context`.


In [ ]:
display(lofo_family["best"])
display(lofo_family["r2_pivot"])
save_table(lofo_family["best"], table_dir, "fmiss_lofo_family_preference_best")
save_table(lofo_family["r2_pivot"].reset_index(), table_dir, "fmiss_lofo_family_preference_r2")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

family_r2 = lofo_family["r2_pivot"].copy()
fig, ax = plt.subplots(figsize=(7.2, 3.8))
norm = mcolors.TwoSlopeNorm(vmin=-1.10, vcenter=0.0, vmax=0.85)
im = ax.imshow(family_r2.to_numpy(dtype=float), cmap="RdYlGn", norm=norm, aspect="auto")
ax.grid(False)
ax.set_xticks(range(family_r2.shape[1]))
ax.set_xticklabels(family_r2.columns.tolist(), rotation=24, ha="right")
ax.set_yticks(range(family_r2.shape[0]))
ax.set_yticklabels([name.replace("PAIRED_", "") for name in family_r2.index.tolist()])
ax.set_title("Mean held-out-family `R²` across the main compared `fmiss` models\n(20-seed leave-one-family-out aggregate)", pad=10)
for row_idx, family_name in enumerate(family_r2.index):
    for col_idx, label in enumerate(family_r2.columns):
        value = family_r2.loc[family_name, label]
        if pd.notna(value):
            text_color = "white" if abs(value) >= 0.72 else "black"
            ax.text(col_idx, row_idx, f"{value:.2f}", ha="center", va="center", fontsize=8, color=text_color)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Mean R²")
fig.tight_layout()
save_figure(fig, fig_dir, "fmiss_lofo_family_heatmap")
fig


### Additional recording-disjoint diagnostics

The plots below remain useful as supporting diagnostics, but the thesis-facing family comparison above is the leave-one-family-out view.


In [ ]:
display(waves["family"])
display(waves["recording"].head(20))
save_table(waves["family"], table_dir, "fmiss_family_wave_summary")
save_table(waves["recording"], table_dir, "fmiss_recording_wave_summary")
fig, _, family_pivot = plot_group_metric(waves["family"], group_col="study_set", metric="r2", title="`fmiss` family R² by model wave\n(paired-family summaries from 20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fmiss_family_r2_by_wave")
save_table(family_pivot, table_dir, "fmiss_family_r2_pivot")
fig


In [ ]:
recording_mean = (
    waves["recording"]
    .groupby(["label", "recording_key"], as_index=False)["r2"]
    .mean()
)
top_rec, bottom_rec = build_extreme_groups_table(recording_mean[recording_mean["label"] == REDUCED_LATENT_CONTEXT_LABEL], group_col="recording_key", metric="r2", top_n=12)
focus_recordings = pd.concat([top_rec, bottom_rec], ignore_index=True)["recording_key"].drop_duplicates().tolist()
recording_focus = waves["recording"][waves["recording"]["recording_key"].isin(focus_recordings)].copy()
display(top_rec)
display(bottom_rec)
save_table(top_rec, table_dir, "fmiss_top_recordings")
save_table(bottom_rec, table_dir, "fmiss_worst_recordings")
fig, _, rec_pivot = plot_group_metric(recording_focus, group_col="recording_key", metric="r2", title="`fmiss` recording R² by model wave\n(top/bottom recordings from 20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fmiss_top_recording_r2_by_wave")
save_table(rec_pivot, table_dir, "fmiss_recording_r2_pivot")
fig


## 4. Dedicated reduced-latent context stack diagnostics

This is the structural comparison that matters most for the `fmiss` story: the final model still behaves like a conservative context-first correction model rather than a dramatic architecture shift.


In [ ]:
context_compare = pd.DataFrame([
    {
        "model": "Paired-only context",
        "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_context", "r2"].iloc[0]),
        "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_context", "mae"].iloc[0]),
    },
    {
        "model": "Reduced-latent context",
        "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_reduced_latent", "r2"].iloc[0]),
        "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_reduced_latent", "mae"].iloc[0]),
    },
])
display(context_compare)
save_table(context_compare, table_dir, "fmiss_context_progress_comparison")
fig, _ = plot_prediction_scatter(winner_diag["predictions"], title="Best `fmiss` model: observed vs predicted")
save_figure(fig, fig_dir, "fmiss_winner_prediction_scatter")
fig


In [ ]:
milestone = pd.DataFrame([
    {"stage": "1  sanity floor", "model": "Dummy paired mean", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_dummy", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_dummy", "mae"].iloc[0])},
    {"stage": "2  paired baseline", "model": "Paired-only raw", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_raw", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_raw", "mae"].iloc[0])},
    {"stage": "3  source transfer", "model": "Hybrid-only XGBoost", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_hybrid_only", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_hybrid_only", "mae"].iloc[0])},
    {"stage": "4  paired context", "model": "Paired-only context", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_context", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_paired_context", "mae"].iloc[0])},
    {"stage": "5  reduced latent", "model": "Reduced-latent context", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_reduced_latent", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fmiss_reduced_latent", "mae"].iloc[0])},
])
display(milestone)
save_table(milestone, table_dir, "fmiss_milestone_trajectory")


In [ ]:
display(Markdown(
    f"""
## Key takeaways

- The best `fmiss` path remained **context-first** and relatively conservative.
- The dedicated reduced-latent context stack reaches correlation **{winner_diag['correlation']:.3f}** on the pooled holdout predictions from the 20 recording-disjoint seeds.
- The main retained gain is incremental rather than dramatic: reduced latent improves on the simpler paired-context model, but `fmiss` remains much harder than `fpos`.
"""
))
